In [0]:
BASE = "s3://mimic-iv-datasets/mimic-iv-dataset-3.1"
HOSP_PATH = f"{BASE}/hosp"
ICU_PATH = f"{BASE}/icu"

CATALOG = "workspace"
BRONZE = "mimic_iv_3_1_bronze"
CATALOG_PATH = f"{CATALOG}.{BRONZE}"

In [0]:
composite_keys = {
    "hosp_admissions": ["hadm_id"],
    "hosp_d_hcpcs": ["code"],
    "hosp_d_icd_diagnoses": ["icd_code", "icd_version"],
    "hosp_d_icd_procedures": ["icd_code", "icd_version"],
    "hosp_d_labitems": ["itemid"],
    "hosp_diagnoses_icd": ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "hosp_drgcodes": ["subject_id", "hadm_id", "drg_type", "drg_code"],
    "hosp_emar": ["emar_id"],
    "hosp_emar_detail": ["emar_id", "parent_field_ordinal"], # (4917 Hard Duplicates)
    "hosp_hcpcsevents": ["subject_id", "hadm_id", "hcpcs_cd", "seq_num"],
    "hosp_labevents": ["labevent_id"],
    "hosp_microbiologyevents": ["microevent_id"],
    "hosp_omr": ["subject_id", "chartdate", "seq_num", "result_name"],
    "hosp_patients": ["subject_id"],
    "hosp_pharmacy": ["pharmacy_id"],
    "hosp_poe": ["poe_id"],
    "hosp_poe_detail": ["poe_id", "field_name", "field_value"],
    "hosp_prescriptions": ["subject_id", "hadm_id", "pharmacy_id", "poe_id", "starttime", "stoptime", "drug_type", "drug"],
    "hosp_procedures_icd": ["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
    "hosp_provider": ["provider_id"],
    "hosp_services": ["subject_id", "hadm_id", "transfertime", "prev_service", "curr_service"],
    "hosp_transfers": ["transfer_id"],
    "icu_caregiver": ["caregiver_id"],
    "icu_chartevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime", "value"], # (73 Hard Duplicates)
    "icu_d_items": ["itemid"],
    "icu_datetimeevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "icu_icustays": ["stay_id"],
    "icu_ingredientevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],    
    "icu_inputevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"],    
    "icu_outputevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "charttime", "storetime"],
    "icu_procedureevents": ["subject_id", "hadm_id", "stay_id", "caregiver_id", "itemid", "starttime", "endtime", "storetime", "orderid"]
}

In [0]:
for table_name, key_columns in composite_keys.items():
        # Read table
        table_path = f"{CATALOG_PATH}.{table_name}"
        df = spark.table(table_path)
        
        # Get total row and distinct key counts
        total_rows = df.count()
        unique_rows = df.groupBy(*key_columns).count().count()
        
        # Check if composite key is unique by comparing counts
        is_unique = total_rows == unique_rows

        print(f"""
        Table: {table_name}
        Primary/Composite Key: {", ".join(key_columns)}
        Total Rows: {total_rows} - Unique Rows: {unique_rows} ({"No Duplicates" if is_unique else "Found Duplicated values"})
        Duplicated Count: {total_rows - unique_rows} {(total_rows - unique_rows) / total_rows:.2f}%
              """)
        